In [1]:
import numpy as np
import geopandas as gpd
from shapely.geometry import Point

# Jakarta bounding box (lon/lat)
BBOX = (106.69, -6.37, 106.97, -6.09)
CATEGORIES = ["cafe", "restaurant", "shop"]


def generate_points(n=50, seed=42):
    """Random POIs in Jakarta as GeoDataFrame (EPSG:4326)."""
    rng = np.random.default_rng(seed)
    lon = rng.uniform(BBOX[0], BBOX[2], n)
    lat = rng.uniform(BBOX[1], BBOX[3], n)
    category = rng.choice(CATEGORIES, n)
    return gpd.GeoDataFrame(
        {
            "name": [f"POI-{i:02d}" for i in range(n)],
            "category": category,
            "rating": rng.uniform(1, 5, n).round(1),
        },
        geometry=[Point(x, y) for x, y in zip(lon, lat)],
        crs="EPSG:4326",
    )


def analyze_and_visualize(gdf, buffer_m=1000):
    """Simple analyses + folium map via GeoDataFrame.explore()."""
    # metric CRS: UTM 48S
    g = gdf.to_crs(32748).copy()

    # 1. distance to centroid of all points
    centroid = g.geometry.union_all().centroid
    g["dist_to_center_m"] = g.distance(centroid).round(0)

    # 2. nearest neighbour distance
    nn = g.sjoin_nearest(g[["geometry"]], distance_col="nn_dist_m", exclusive=True)
    nn = nn[~nn.index.duplicated()]
    g["nn_dist_m"] = nn["nn_dist_m"].round(0)

    # 3. buffer around each cafe, count POIs inside (excluding itself)
    cafes = g[g.category == "cafe"].copy()
    cafes["geometry"] = cafes.buffer(buffer_m)
    joined = gpd.sjoin(g[["geometry"]], cafes[["name", "geometry"]], predicate="within")
    counts = joined.groupby("name").size() - 1
    cafes["poi_in_buffer"] = cafes["name"].map(counts).fillna(0).astype(int)

    # 4. convex hull per category
    hulls = g.dissolve(by="category").convex_hull.reset_index()
    hulls.columns = ["category", "geometry"]
    hulls = gpd.GeoDataFrame(hulls, geometry="geometry", crs=g.crs)

    # summary
    print(g.groupby("category")[["rating", "dist_to_center_m", "nn_dist_m"]].mean().round(1))
    print(f"\nBusiest cafe buffer: {cafes.loc[cafes.poi_in_buffer.idxmax(), ['name', 'poi_in_buffer']].to_dict()}")

    # map (explore on EPSG:4326 data, layers chained via m=)
    m = hulls.to_crs(4326).explore(
        column="category", cmap="Set2", legend=False, name="Convex hull by category",
        style_kwds={"fillOpacity": 0.08}, tiles="OpenStreetMap", control_scale=True,
    )
    m = cafes.to_crs(4326).explore(
        m=m, column="poi_in_buffer", cmap="YlOrRd", legend=True,
        name=f"Cafe {buffer_m} m buffer", style_kwds={"fillOpacity": 0.25, "weight": 1},
    )
    m = g.to_crs(4326).explore(
        m=m, column="category", cmap="Set1", name="POI",
        tooltip=["name", "category", "rating", "dist_to_center_m", "nn_dist_m"],
        marker_kwds={"radius": 6}, legend=True,
    )
    gpd.GeoDataFrame({"name": ["Centroid"]}, geometry=[centroid], crs=g.crs).to_crs(4326).explore(
        m=m, color="black", name="Centroid", marker_kwds={"radius": 9}, tooltip="name",
    )
    import folium
    folium.LayerControl().add_to(m)
    return m


gdf = generate_points(n=50)
analyze_and_visualize(gdf)


            rating  dist_to_center_m  nn_dist_m
category                                       
cafe           2.6           11743.7     2945.0
restaurant     3.1           11305.2     1836.0
shop           3.0            9859.9     2133.8

Busiest cafe buffer: {'name': 'POI-20', 'poi_in_buffer': 1}
